In [ ]:
# Load R magic extension for Python Jupyter kernel (Kaggle / Colab support)
try:
    %load_ext rpy2.ipython
except Exception as e:
    print("Note on rpy2 initialization:", e)

# Layer 1: Feature Engineered Extreme ESI Logistic Regressor (`models/lr_extreme.ipynb`)

This notebook trains a **Layer 1 Multinomial Logistic Regressor** using **13 Clinical Feature Engineered Inputs** with **Strict Complete Case Filtering**:
- **Feature Engineering Inputs**: Includes age, gender, breathing difficulty chief complaint, dyspnea flags (o2 < 90, 90-94), bradypnea/tachypnea flags (rr < 10, > 30), hypotension/hypertension flags (sbp <= 90, > 220), and bradycardia/tachycardia flags (hr < 40, 40-60, > 150, 100-150).
- **Complete Case Filtering**: Removes any row with at least one NULL/NA feature across all splits.
- **Target Output**: Predicts whether a patient is **ESI 1** (Highest Acuity), **ESI 5** (Lowest Acuity), or **neither** (Intermediate ESI 2, 3, 4).
- **User-Configurable Class Weights Dictionary**: Allows custom per-class weights (`c("1" = 1.0, "5" = 1.0, "neither" = 1.0)`) directly editable in Step 4.
- **Evaluation Metrics**: **Accuracy**, **Precision**, **Recall (Sensitivity)**, **PR-AUC**, **Multi-Class ROC-AUC**, and **Actual vs Predicted Class Counts**.
- **Model Export**: Saved to `deploy/lr_extreme_model.rds`.

In [ ]:
%%R
# ---------------------------------------------------------
# Step 1: Load Required Libraries & Parse Configuration JSON
# ---------------------------------------------------------
library(jsonlite)
library(caret)
library(nnet)
library(dplyr)
library(ggplot2)
library(pROC)

config_path <- "../config/triage_conf.json"
if (!file.exists(config_path)) {
  config_path <- "config/triage_conf.json"
}

config <- fromJSON(config_path)

cat("=== Configuration Loaded from config/triage_conf.json ===\n")
cat("Data Source Path:", config$path$data_source, "\n")
cat("Target Column:   ", config$classes$target_col, "\n")
cat("Test Size:       ", config$training$test_size, "\n")
cat("Val Size:        ", config$training$val_size, "\n")
cat("Random State:    ", config$training$random_state, "\n")

In [ ]:
%%R
# ---------------------------------------------------------
# Step 2: Load Data, Construct 13 Feature Engineered Inputs, & Apply Complete Case Analysis
# ---------------------------------------------------------
set.seed(config$training$random_state)

data_file <- config$path$data_source
if (!file.exists(data_file) && file.exists(paste0("../", data_file))) {
  data_file <- paste0("../", data_file)
}

cat("Loading dataset from:", data_file, "...\n")

data_env <- new.env()
load(data_file, envir = data_env)

df_names <- ls(data_env)[sapply(ls(data_env), function(x) is.data.frame(get(x, envir = data_env)))]
df_sizes <- sapply(df_names, function(x) nrow(get(x, envir = data_env)))
data_obj_name <- df_names[which.max(df_sizes)]
cat(sprintf("Selected main dataset object: '%s' (%d rows)\n", data_obj_name, max(df_sizes)))

raw_df <- get(data_obj_name, envir = data_env)
target_col <- config$classes$target_col

gender_vec <- if ("gender" %in% names(raw_df)) ifelse(as.character(raw_df$gender) == "Male", 1, 0) else 0
cc_bd_vec  <- if ("cc_breathingdifficulty" %in% names(raw_df)) ifelse(!is.na(raw_df$cc_breathingdifficulty), raw_df$cc_breathingdifficulty, 0) else 0

# Construct 13 Clinical Feature Engineered Inputs
df_feng <- data.frame(
  age                     = raw_df$age,
  gender                  = gender_vec,
  cc_breathingdifficulty  = cc_bd_vec,
  is_dyspnea_total        = ifelse(!is.na(raw_df$triage_vital_o2) & raw_df$triage_vital_o2 < 90, 1, 0),
  is_dyspnea_moderate     = ifelse(!is.na(raw_df$triage_vital_o2) & raw_df$triage_vital_o2 >= 90 & raw_df$triage_vital_o2 < 94, 1, 0),
  is_bradypnea            = ifelse(!is.na(raw_df$triage_vital_rr) & raw_df$triage_vital_rr < 10, 1, 0),
  is_tachypnea            = ifelse(!is.na(raw_df$triage_vital_rr) & raw_df$triage_vital_rr > 30, 1, 0),
  is_hypotension          = ifelse(!is.na(raw_df$triage_vital_sbp) & raw_df$triage_vital_sbp <= 90, 1, 0),
  is_hypertension         = ifelse(!is.na(raw_df$triage_vital_sbp) & raw_df$triage_vital_sbp > 220, 1, 0),
  is_bradycardia_total    = ifelse(!is.na(raw_df$triage_vital_hr) & raw_df$triage_vital_hr < 40, 1, 0),
  is_bradycardia_moderate = ifelse(!is.na(raw_df$triage_vital_hr) & raw_df$triage_vital_hr >= 40 & raw_df$triage_vital_hr < 60, 1, 0),
  is_tachycardia_total    = ifelse(!is.na(raw_df$triage_vital_hr) & raw_df$triage_vital_hr > 150, 1, 0),
  is_tachycardia_moderate = ifelse(!is.na(raw_df$triage_vital_hr) & raw_df$triage_vital_hr > 100 & raw_df$triage_vital_hr <= 150, 1, 0)
)

raw_esi <- as.character(raw_df[[target_col]])
df_feng[[target_col]] <- raw_esi

initial_rows <- nrow(df_feng)
df <- na.omit(df_feng)
cat(sprintf("Complete Case Filtering: Removed %d rows with NULL/NA features (Remaining complete rows: %d)\n",
            initial_rows - nrow(df), nrow(df)))

# Target for Layer 1: '1', '5', or 'neither'
raw_esi_complete <- as.character(df[[target_col]])
df$target_layer1 <- factor(ifelse(raw_esi_complete == "1", "1",
                            ifelse(raw_esi_complete == "5", "5", "neither")),
                           levels = c("1", "5", "neither"))

cat(sprintf("Layer 1 Feature Engineered Dataset Ready: %d rows x %d cols\n", nrow(df), ncol(df)))
cat("Layer 1 Target Distribution (Complete Cases):\n")
print(table(df$target_layer1))

In [ ]:
%%R
# ---------------------------------------------------------
# Step 3: Stratified Data Partitioning & Feature Scaling
# ---------------------------------------------------------
set.seed(config$training$random_state)

test_size <- config$training$test_size
val_size  <- config$training$val_size

# Stratified Test split (15%)
in_train_val <- createDataPartition(df$target_layer1, p = 1 - test_size, list = FALSE)
train_val_df <- df[in_train_val, ]
test_df      <- df[-in_train_val, ]

# Stratified Validation split (15%)
rel_val_size <- val_size / (1 - test_size)
in_train    <- createDataPartition(train_val_df$target_layer1, p = 1 - rel_val_size, list = FALSE)
train_df    <- train_val_df[in_train, ]
val_df      <- train_val_df[-in_train, ]

# Standardize numeric features (center and scale)
numeric_cols <- names(train_df)[sapply(train_df, is.numeric)]
preproc <- preProcess(train_df[, numeric_cols], method = c("center", "scale"))

train_df[, numeric_cols] <- predict(preproc, train_df[, numeric_cols])
val_df[, numeric_cols]   <- predict(preproc, val_df[, numeric_cols])
test_df[, numeric_cols]  <- predict(preproc, test_df[, numeric_cols])

cat(sprintf("Complete Case Partition sizes:\n  Train: %d rows\n  Val:   %d rows\n  Test:  %d rows\n",
            nrow(train_df), nrow(val_df), nrow(test_df)))

In [ ]:
%%R
# ---------------------------------------------------------
# Step 4: User-Configurable Class Weights Dictionary & Training
# ---------------------------------------------------------
set.seed(config$training$random_state)

# ---------------------------------------------------------
# USER-CONFIGURABLE CLASS WEIGHTS DICTIONARY
# Modify class weights below as desired for '1', '5', and 'neither'
# ---------------------------------------------------------
custom_class_weights <- c(
  "1"       = 1.0,
  "5"       = 10.0,
  "neither" = 1.0
)

cat("User-Configured Class Weights Dictionary:\n")
print(custom_class_weights)

# Map per-sample observation weights using the user dictionary
sample_weights <- as.numeric(custom_class_weights[as.character(train_df$target_layer1)])

feat_names <- setdiff(names(train_df), c(target_col, "target_layer1"))
formula_lr <- as.formula(paste("target_layer1 ~", paste(feat_names, collapse = " + ")))

cat("\nTraining Layer 1 Multinomial Logistic Regressor with Feature Engineered Inputs...\n")
lr_model <- multinom(formula_lr, data = train_df, weights = sample_weights, trace = FALSE, MaxNWts = 5000)

cat("Layer 1 Feature Engineered Logistic Regression training complete!\n")
print(summary(lr_model))

In [ ]:
%%R
# ---------------------------------------------------------
# Step 5: Evaluate Scoring Metrics (Accuracy, Precision, Recall, PR-AUC, ROC-AUC, Class Counts)
# ---------------------------------------------------------
# Function to compute PR-AUC (Precision-Recall Area Under Curve)
calc_pr_auc <- function(actual_binary, prob_positive) {
  tryCatch({
    ord <- order(prob_positive, decreasing = TRUE)
    act_sorted <- (actual_binary[ord] == 1)
    tp <- cumsum(act_sorted)
    fp <- cumsum(!act_sorted)
    n_pos <- sum(act_sorted)
    if (n_pos == 0) return(NA)
    rec <- c(0, tp / n_pos)
    prec <- c(tp[1] / max(1, tp[1] + fp[1]), tp / (tp + fp))
    dx <- diff(rec)
    my <- (prec[-1] + prec[-length(prec)]) / 2
    return(as.numeric(sum(dx * my)))
  }, error = function(e) NA)
}
evaluate_layer1_lr <- function(model, data, set_name) {
  prob_matrix <- predict(model, newdata = data, type = "probs")
  target_classes <- levels(data$target_layer1)
  
  max_idx <- max.col(prob_matrix, ties.method = "first")
  pred_factor <- factor(colnames(prob_matrix)[max_idx], levels = target_classes)
  actual_factor <- factor(data$target_layer1, levels = target_classes)
  
  cm <- confusionMatrix(pred_factor, actual_factor)
  acc <- as.numeric(cm$overall["Accuracy"])
  
  prec_by_class <- cm$byClass[, "Pos Pred Value"]
  rec_by_class  <- cm$byClass[, "Sensitivity"]
  macro_prec    <- mean(prec_by_class, na.rm = TRUE)
  macro_rec     <- mean(rec_by_class,  na.rm = TRUE)
  
  pr_auc_by_class <- numeric(length(target_classes))
  names(pr_auc_by_class) <- target_classes
  for (cls in target_classes) {
    act_bin <- ifelse(actual_factor == cls, 1, 0)
    pr_auc_by_class[cls] <- calc_pr_auc(act_bin, prob_matrix[, cls])
  }
  macro_pr_auc <- mean(pr_auc_by_class, na.rm = TRUE)
  
  roc_auc <- tryCatch({
    as.numeric(pROC::multiclass.roc(actual_factor, prob_matrix)$auc)
  }, error = function(e) NA)
  
  actual_table <- table(actual_factor)
  pred_table   <- table(pred_factor)
  
  diff_vec <- as.numeric(pred_table) - as.numeric(actual_table)
  diff_str <- ifelse(diff_vec >= 0, paste0("+", diff_vec), as.character(diff_vec))
  
  class_comparison <- data.frame(
    Class        = target_classes,
    Actual_Count = as.numeric(actual_table),
    Pred_Count   = as.numeric(pred_table),
    Diff         = diff_str,
    Precision    = round(prec_by_class, 4),
    Recall       = round(rec_by_class, 4),
    PR_AUC       = round(pr_auc_by_class, 4)
  )
  
  cat(sprintf("============================================================\n"))
  cat(sprintf("   FEATURE ENGINEERED LAYER 1 LOGISTIC REGRESSOR - %s SET BENCHMARK\n", toupper(set_name)))
  cat(sprintf("============================================================\n"))
  cat(sprintf("  Overall Accuracy        : %.4f (%.2f%%)\n", acc, acc * 100))
  cat(sprintf("  Macro Precision         : %.4f (%.2f%%)\n", macro_prec, macro_prec * 100))
  cat(sprintf("  Macro Recall (Sens)     : %.4f (%.2f%%)\n", macro_rec, macro_rec * 100))
  cat(sprintf("  Macro PR-AUC            : %.4f\n", macro_pr_auc))
  cat(sprintf("  Multi-Class ROC-AUC     : %.4f\n", roc_auc))
  cat(sprintf("============================================================\n\n"))
  
  cat("Target Class Count Comparison & Performance Summary Table:\n")
  print(class_comparison)
  
  cat("\nFull Confusion Matrix (Rows: Predicted, Columns: Actual):\n")
  print(cm$table)
  cat(sprintf("============================================================\n\n"))
}
# Evaluate on Validation set
evaluate_layer1_lr(lr_model, val_df, "Validation")
# Evaluate on Test set
evaluate_layer1_lr(lr_model, test_df, "Test")

In [ ]:
%%R
# ---------------------------------------------------------
# Step 6: Save Layer 1 Model Artifacts
# ---------------------------------------------------------
deploy_dir <- "../deploy"
if (!dir.exists(deploy_dir)) deploy_dir <- "deploy"
if (!dir.exists(deploy_dir)) dir.create(deploy_dir, recursive = TRUE)

model_path <- file.path(deploy_dir, "lr_extreme_model.rds")
saveRDS(list(model = lr_model, preproc = preproc), file = model_path)
cat("Layer 1 Feature Engineered Extreme Logistic Regressor model saved to:", model_path, "\n")